##Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

##Parameters

In [0]:
CATALOG = "workspace"
SCHEMA = "final"

HEADER_TABLE = f"{CATALOG}.{SCHEMA}.silver_sales_order_header"
DETAIL_TABLE = f"{CATALOG}.{SCHEMA}.silver_sales_order_detail"

CUSTOMER_DIM = f"{CATALOG}.{SCHEMA}.gold_dim_customer"
PRODUCT_DIM = f"{CATALOG}.{SCHEMA}.gold_dim_product"
DATE_DIM = f"{CATALOG}.{SCHEMA}.gold_dim_date"

TARGET_TABLE = f"{CATALOG}.{SCHEMA}.gold_fact_sales"

##Read Sources

In [0]:
header_df = spark.table(HEADER_TABLE)

detail_df = spark.table(DETAIL_TABLE)

customer_dim = (
    spark.table(CUSTOMER_DIM)
    .filter("is_current = 1")
)

product_dim = (
    spark.table(PRODUCT_DIM)
    .filter("is_current = 1")
)

date_dim = spark.table(DATE_DIM)

##Join Header + Detail

In [0]:
sales_df = (
    detail_df.alias("d")
    .join(
        header_df.alias("h"),
        "SalesOrderID"
    )
)

##Customer Lookup

In [0]:
sales_df = (
    sales_df.alias("f")
    .join(
        broadcast(
            customer_dim.select(
                "CustomerID",
                "Customer_SK"
            )
        ),
        "CustomerID",
        "left"
    )
)

##Product Lookup

In [0]:
sales_df = (
    sales_df.alias("f")
    .join(
        broadcast(
            product_dim.select(
                "ProductID",
                "Product_SK"
            )
        ),
        "ProductID",
        "left"
    )
)

##Date Lookup

In [0]:
sales_df = sales_df.withColumn(
    "OrderDate_SK",
    F.date_format(
        F.to_date("OrderDate"),
        "yyyyMMdd"
    ).cast("int")
)

##Build Fact Dataset

In [0]:
fact_df = (
    sales_df
    .select(
        "SalesOrderID",
        "SalesOrderDetailID",
        "Customer_SK",
        "Product_SK",
        "OrderDate_SK",
        "OrderQty",
        "UnitPrice",
        "NetSalesAmount"
    )
)

##Derived Measures

In [0]:
fact_df = fact_df.withColumn(
    "LineAmount",
    F.round(
        F.col("OrderQty") *
        F.col("UnitPrice"),
        2
    )
)

##Audit Columns

In [0]:
fact_df = (
    fact_df
    .withColumn(
        "LoadDate",
        F.current_timestamp()
    )
)

##Create Fact Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.final.gold_fact_sales
(
    Sales_SK BIGINT GENERATED ALWAYS AS IDENTITY,

    SalesOrderID BIGINT,

    SalesOrderDetailID BIGINT,

    Customer_SK BIGINT,

    Product_SK BIGINT,

    OrderDate_SK INT,

    OrderQty INT,

    UnitPrice DECIMAL(18,2),

    NetSalesAmount DECIMAL(18,2),

    LineAmount DECIMAL(18,2),

    LoadDate TIMESTAMP
)
USING DELTA

##Load Fact

In [0]:
fact_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(TARGET_TABLE)

##Optimize

In [0]:
%sql
OPTIMIZE workspace.final.gold_fact_sales
ZORDER BY (
    Customer_SK,
    Product_SK,
    OrderDate_SK
)

##Validation

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.final.gold_fact_sales

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.final.silver_sales_order_detail

##Surrogate Key Validation

####Customer lookup success:

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.final.gold_fact_sales
WHERE Customer_SK IS NULL

####Product lookup success:

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.final.gold_fact_sales
WHERE Product_SK IS NULL